<a href="https://colab.research.google.com/github/Sreekar-DS/Unsupervised-ML---Myntra-Customer-Segmentation/blob/main/Colab%20Notebook/Unsupervised_ML_Online_Retail_Customer_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Unsupervised Machine Learning Project**  
## **Online Retail Customer Segmentation for Myntra Gifts Ltd.**

##### **Project Type** - Unsupervised Machine Learning / Customer Segmentation  
##### **Contribution** - Individual  
##### **Student Name** - Tarun Sreekar Parasa
##### **Video Presentation Link** - https://drive.google.com/file/d/1pXj9M_xgWpNe8ryVhZdLsnFh3Jfv_Aak/view?usp=sharing

# **Project Summary**

This project applies unsupervised machine learning to an online retail transaction dataset belonging to Myntra Gifts Ltd., a UK-based online giftware business. The dataset contains invoice-level transaction records with product, quantity, price, customer, date, and country information. Since there is no predefined target variable, the main objective is to discover natural customer groups based on purchasing behavior and convert those groups into practical business strategies.

The project follows a complete unsupervised learning workflow. First, exploratory data analysis is performed to understand the structure of the data, transaction period, missing values, duplicate records, cancelled invoices, product popularity, country-wise demand, order value patterns, and monthly purchasing trends. This stage helps identify issues such as missing customer IDs, negative quantities caused by cancelled transactions, skewed monetary values, and high-value outliers.

After EDA, the data is cleaned by removing records without CustomerID, cancelled invoices, non-positive quantities, non-positive prices, and duplicate rows. A total purchase amount feature is created by multiplying quantity with unit price. The transaction-level data is then transformed into a customer-level analytical dataset using RFM-style features: Recency, Frequency, Monetary value, average order value, total quantity purchased, and number of distinct products bought. These features are selected because they describe how recently a customer purchased, how often they buy, how much they spend, and how broad their product interest is.

The feature engineering stage also includes outlier handling and logarithmic transformation for highly skewed variables. Standard scaling is applied because clustering algorithms such as K-Means and hierarchical clustering are distance-based and can be strongly influenced by feature scale. Several values of K are tested using elbow and silhouette methods to estimate a suitable number of clusters. K-Means clustering is then used as the final model because it is simple, interpretable, efficient on customer-level data, and suitable for business segmentation.

Finally, clusters are visualized using PCA and profiled using median values for key customer metrics. Each cluster is interpreted in business language, such as high-value loyal customers, recent moderate customers, low-value/infrequent customers, and dormant customers. The output can help stakeholders design segment-specific strategies including retention campaigns, win-back offers, product recommendations, inventory planning, and personalized marketing.

# **Problem Statement**

Myntra Gifts Ltd. wants to better understand its customer base using historical online retail transactions. The business has invoice-level purchase data but does not have predefined customer labels such as loyal, inactive, high-value, or low-value customers. Therefore, the goal is to use unsupervised machine learning to segment customers based on purchasing behavior.

The project aims to answer the following business questions:

1. Which customer groups naturally exist in the transaction data?
2. How do customers differ in terms of recency, frequency, monetary value, quantity purchased, and product variety?
3. What is the ideal number of customer clusters?
4. How can each customer segment be interpreted for marketing, retention, inventory, and pricing strategies?

# **Business Context**

Myntra is a leading Indian fashion e-commerce company known for a wide range of clothing, accessories, and lifestyle products. In this project context, the dataset represents online retail operations for Myntra Gifts Ltd., a UK-based division specializing in unique all-occasion giftware. The data contains transactions from **01 December 2010 to 09 December 2011** and provides a detailed view of international online retail activity.

The purpose of this analysis is to extract useful business insights and build meaningful customer segments. These segments can help the company identify purchasing trends, evaluate product performance, understand customer behavior, optimize pricing strategies, and streamline inventory management.

# **Project Architecture**

The project follows this workflow:

1. **EDA** - Understand the business problem, visualize relationships, analyze product popularity, country-wise orders, and customer spending behavior.
2. **Clean-up** - Handle missing values, duplicate rows, cancelled invoices, invalid quantities, invalid prices, and outliers.
3. **Feature Engineering** - Create customer-level RFM and behavioral features.
4. **Preprocessing** - Apply outlier capping, log transformation, and feature scaling.
5. **Model Implementation** - Test the ideal number of clusters using elbow and silhouette charts, then apply K-Means clustering.
6. **Model Explainability** - Interpret how many clusters exist, explain what each cluster means, and suggest stakeholder strategies.

# **Dataset Description**

| Field | Description |
|---|---|
| InvoiceNo | Invoice number |
| StockCode | Product stock code |
| Description | Product description |
| Quantity | Quantity bought |
| InvoiceDate | Date and time of invoice |
| UnitPrice | Price per unit |
| CustomerID | Unique customer ID |
| Country | Customer/order location |

In [2]:
# ==============================
# 1. Import Libraries
# ==============================

import warnings
warnings.filterwarnings('ignore')

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA

from scipy.cluster.hierarchy import linkage, dendrogram

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

In [3]:
# ==============================
# 2. Data Loading Helper
# ==============================

DATA_URL = "https://drive.google.com/file/d/1spGIIVJQ-QlCjlSHnf7JwzCgkIbK8jYS/view?usp=sharing"


def convert_google_drive_url(url):
    """Convert a Google Drive sharing URL into a direct download URL."""
    if not url:
        return url
    if "drive.google.com" in url and "/file/d/" in url:
        file_id = re.search(r"/file/d/([^/]+)", url).group(1)
        return f"https://drive.google.com/uc?id={file_id}"
    return url


def load_online_retail_data(data_url=""):
    """Load the retail CSV from URL."""
    if data_url:
        url = convert_google_drive_url(data_url)
        print(f"Loading data from URL: {url}")
        return pd.read_csv(url, encoding="ISO-8859-1")

    raise ValueError("DATA_URL is empty. Please provide a Google Drive link to the CSV file.")

raw_df = load_online_retail_data(DATA_URL)
print("Dataset shape:", raw_df.shape)
raw_df.head()

Loading data from URL: https://drive.google.com/uc?id=1spGIIVJQ-QlCjlSHnf7JwzCgkIbK8jYS
Dataset shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,"17,850.00",United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,"17,850.00",United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,"17,850.00",United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,"17,850.00",United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,"17,850.00",United Kingdom


In [4]:
# ==============================
# 3. Basic Data Understanding
# ==============================

display(raw_df.info())
display(raw_df.describe(include='all').T)

missing_summary = pd.DataFrame({
    'Missing_Count': raw_df.isna().sum(),
    'Missing_Percentage': (raw_df.isna().mean() * 100).round(2)
}).sort_values('Missing_Percentage', ascending=False)

missing_summary

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


None

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
InvoiceNo,541909,25900,573585,1114,NaN,NaN,NaN,NaN,NaN,NaN,NaN
StockCode,541909,4070,85123A,2313,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,540455,4223,WHITE HANGING HEART T-LIGHT HOLDER,2369,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,"541,909.00",NaN,NaN,NaN,9.55,218.08,"-80,995.00",1.00,3.00,10.00,"80,995.00"
InvoiceDate,541909,23260,10/31/11 14:41,1114,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UnitPrice,"541,909.00",NaN,NaN,NaN,4.61,96.76,"-11,062.06",1.25,2.08,4.13,"38,970.00"
CustomerID,"406,829.00",NaN,NaN,NaN,"15,287.69","1,713.60","12,346.00","13,953.00","15,152.00","16,791.00","18,287.00"
Country,541909,38,United Kingdom,495478,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,Missing_Count,Missing_Percentage
CustomerID,135080,24.93
Description,1454,0.27
StockCode,0,0.00
InvoiceNo,0,0.00
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
Country,0,0.00


In [5]:
# Check duplicate records and cancelled invoices
print("Duplicate rows:", raw_df.duplicated().sum())
print("Cancelled invoice rows:", raw_df['InvoiceNo'].astype(str).str.startswith('C').sum())
print("Unique invoices:", raw_df['InvoiceNo'].nunique())
print("Unique products:", raw_df['StockCode'].nunique())
print("Unique customers before cleaning:", raw_df['CustomerID'].nunique())
print("Unique countries:", raw_df['Country'].nunique())

Duplicate rows: 5268
Cancelled invoice rows: 9288
Unique invoices: 25900
Unique products: 4070
Unique customers before cleaning: 4372
Unique countries: 38


In [6]:
# Convert InvoiceDate and inspect transaction period
raw_df['InvoiceDate'] = pd.to_datetime(raw_df['InvoiceDate'], format='%m/%d/%y %H:%M', errors='coerce')
print("Start date:", raw_df['InvoiceDate'].min())
print("End date:", raw_df['InvoiceDate'].max())

Start date: 2010-12-01 08:26:00
End date: 2011-12-09 12:50:00
